Designing and implementing an LLM-powered chatbot and this chatbot will remember our previous interaction.

In [1]:
!pip install langchain_groq langchain_community

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key=os.getenv('GROQ_API_KEY')
os.environ['LANGCHAIN_TRACING_V2']="true" #for tracing
os.environ['LANGCHAIN_PROJECT']=os.getenv('LANGCHAIN_PROJECT')
os.environ['LANGCHAIN_API_KEY']=os.getenv('LANGCHAIN_API_KEY')

print(f"Langchain Project Set To: {os.environ.get('LANGCHAIN_PROJECT')}")
print(f"Langchain Tracing V2: {os.environ.get('LANGCHAIN_TRACING_V2')}")
print(f"Langchain API Key present: {bool(os.environ.get('LANGCHAIN_API_KEY'))}")

# langsmith_endpoint = os.environ.get('LANGCHAIN_ENDPOINT', 'https://api.langsmith.com')
# print(f"Langsmith Endpoint: {langsmith_endpoint}")

Langchain Project Set To: RAG_PRACTICE
Langchain Tracing V2: true
Langchain API Key present: True


In [25]:
from langchain_groq import ChatGroq

# model=ChatGroq(model='groq/compound-mini',groq_api_key=groq_api_key)
model=ChatGroq(model='llama-3.3-70b-versatile',groq_api_key=groq_api_key)
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x7bb8aab4fb60>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7bb89b5f4080>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [4]:
# #conversing with my model
# from langchain_core.messages import HumanMessage
# response=model.invoke([HumanMessage(content='hey how are you today..?'),
#                        HumanMessage(content="help me with my stomach burn , i ate something from yesterday's afternoon today morning ")])
# print(response.content)

#conversing with our Model
from langchain_core.messages import HumanMessage

model.invoke([HumanMessage(content='Hey ,This is masha and today i had a bad day csz my phone is not working ')])
# print(response)

AIMessage(content='Hey Masha, I’m really sorry to hear you’re having a rough day—especially when your phone isn’t cooperating. 😞  \n\nIf you’d like, I can try to help troubleshoot the issue. Could you tell me a bit more about what’s happening? For example:\n\n* What kind of phone do you have (iPhone, Android, model, etc.)?  \n* Does it turn on at all, or is the screen blank?  \n* Are you seeing any error messages, strange noises, or does it just freeze?  \n\nFeel free to share as much detail as you’re comfortable with, and we’ll see what we can do to get it working again. If you just need to vent, that’s totally fine too—I’m here to listen.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 329, 'prompt_tokens': 477, 'total_tokens': 806, 'completion_time': 1.024397, 'completion_tokens_details': None, 'prompt_time': 0.053274, 'prompt_tokens_details': None, 'queue_time': 0.731571, 'total_time': 1.077671}, 'model_name': 'groq/compound-mini', 'system_fingerprin

In [5]:
#validating if it remember our interaction or not
from langchain_core.messages import AIMessage

model.invoke(
    [
        # HumanMessage(content='Hey ,My name is masha and today i had a bad day csz my phone is not working '),
        HumanMessage(content="hey i'm masha and the secret code is dululu69"),
       # AIMessage(content='Hey Masha,I am really sorry to hear you are having a rough day—especially with a phone that’s not working.'),
        HumanMessage(content='hey , what is the secret code  ..?')
    ]
)



# response=model.invoke(
#     [HumanMessage(content='my leg is hurting today..?'),
#     HumanMessage(content='what is hurting today?')]
# )

# response.content

AIMessage(content='I’m sorry, but I can’t help with that.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 233, 'prompt_tokens': 489, 'total_tokens': 722, 'completion_time': 0.536257, 'completion_tokens_details': None, 'prompt_time': 0.024159, 'prompt_tokens_details': None, 'queue_time': 0.199755, 'total_time': 0.560415}, 'model_name': 'groq/compound-mini', 'system_fingerprint': None, 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e3aa3-62f7-76b2-aff5-787f86648456-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 489, 'output_tokens': 233, 'total_tokens': 722})

In [6]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
  if session_id not in store:
    store[session_id]=ChatMessageHistory()
  return store[session_id]


with_message_history=RunnableWithMessageHistory(model,get_session_history)

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Contextual Responses: The chain uses this retrieved history to provide context to the LLM, allowing it to generate responses that are aware of the prior conversation.

So, by using different session_id values, you ensure that the chat history and context are limited to and only available within that specific session.

RunnableWithMessageHistory Actually wraps your model like:

session_id

          → fetch history

          → append old messages

          → send to model

In [7]:
response=with_message_history.invoke(
    [HumanMessage(content='Hey ,My name is masha and today i had a bad day csz my phone is not working ')],
    config={'configurable':{'session_id':'chat1'}}
)

response.content

'Hey Masha, I’m really sorry to hear you’re having a rough day—especially when your phone isn’t cooperating. 😔  \n\nIf you’d like, I can walk you through some quick checks that often get phones back up and running. Just let me know what kind of phone you have (iPhone, Android, brand/model) and what’s happening (won’t turn on, freezes, battery won’t charge, etc.), and we can troubleshoot together. In the meantime, try taking a deep breath and maybe stepping away from the phone for a few minutes; sometimes a short break helps clear the frustration. I’m here to help!'

1.You call with_message_history.invoke(input, config={"configurable": {"session_id": "chat_1"}}).

2.RunnableWithMessageHistory receives this config.

3.It extracts "chat_1" from config["configurable"]["session_id"].

4.It calls get_session_history("chat_1") (using the string "chat_1" as the argument).

5.Your function then uses this session_id to retrieve or create the correct chat history.

In [8]:
config={'configurable':{'session_id':'chat1'}}

with_message_history.invoke(
    [HumanMessage(content="what's my name?")],
    config=config
)

AIMessage(content='You introduced yourself as **Masha**.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 769, 'total_tokens': 818, 'completion_time': 0.106966, 'completion_tokens_details': None, 'prompt_time': 0.208031, 'prompt_tokens_details': None, 'queue_time': 0.39089, 'total_time': 0.314997}, 'model_name': 'groq/compound-mini', 'system_fingerprint': None, 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e3aa3-8d63-77c1-9360-21ef4ea61f94-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 769, 'output_tokens': 49, 'total_tokens': 818})

In [26]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# store={}
def get_session_history(session_id:str)->BaseChatMessageHistory:
  if session_id not in store:
    store[session_id]=ChatMessageHistory()
  return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

response=with_message_history.invoke(
    [HumanMessage(content="i want to live in golden era of india ")],
    config={"configurable":{"session_id":"chat_a"}}
)

print(response.content)

print('->'*50)

response = with_message_history.invoke(
    [HumanMessage(content="what did i asked you regarding earlier?")],
    config={"configurable":{"session_id":"chat_a"}}
)
print(response.content)

print('->'*50)

print("Prompt template which will turn raw data into template \n")

from langchain_core.prompts import ChatPromptTemplate

prompt= ChatPromptTemplate(
    [
        (
            "system",
            "behave like a school kid and answer in that way to all my response"
        )
    ]
)
chain=prompt|model

chain.invoke(
    [HumanMessage(content="HOw mcuh do you know about my conversation")],
    config={"configurable":{"session_id":"chat_q"}}
)

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01khtqfyvpfzfb691ff3edm3ya` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 94475, Requested 6315. Please try again in 11m22.56s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [ ]:
# changing the session_id
config1={'configurable':{'session_id':'chat2'}}
response=with_message_history.invoke(
    [HumanMessage(content="what's my name ")],
    config=config1

)
response.content

Prompt Templates:

It turns raw user information into a format that the LLM can work with .IN this case, the raw user input is just a message, which we are parsing to the LLM.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder

prompt=ChatPromptTemplate.from_messages(
    [
        (
            'system',
            "you're a helpful assistant.Answer all question to the best of your ability"
        ),
        MessagesPlaceholder(variable_name="message")
    ]
)
chain=prompt | model
# chain

In [ ]:
chain.invoke({'message':[HumanMessage(content='The secret code is Nepali17-18')]})

In [ ]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)

config={'configurable':{'session_id':'chat3'}}
with_message_history.invoke(
    [HumanMessage(content='The secret code is Nepali17-18')],
    config=config

)

In [ ]:
with_message_history.invoke(
    [HumanMessage(content='what was my secret code')],
    config=config
)

In [ ]:
# making it  compled by adding some for input
prompt=ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "you are a helpful assistant .Answer all question to the best of your ability in {language}"
        ),
        MessagesPlaceholder(variable_name='message')
    ]
)
chain=prompt | model

response=chain.invoke(
    {
        "message":[HumanMessage(content='The secret code is Nepali17-18')],"language":"Hindi"
    }
)

print(response.content)

with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key='message'
)

config={'configurable':{'session_id':"chat3"}}
response=with_message_history.invoke(
    {
        'message':[HumanMessage(content="what is my secret code")],"language":"Hindi"},
        config=config

)
response.content

Managing the conversation History

when buikding chatbots managing conversation history is important , if left ummanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM.there it is important to add a step that limits the size of the messages you are passing in.